# Module 1: Regression

This learning module develops both linear and polynomial regression from the ground up. All code is included at the end of the module.

## Problem Formulation

Suppose we have a finite sequence $\mathcal{D} = (\mathbf{x}_n)^N_{n=1} \subseteq \mathbb{R}^K$ with $K \in \mathbb{N}$. We will write elements of this sequence as $\mathbf{x}_n = (x_{n,1}, \ldots, x_{n,K})$, $1 \le n \le N$. Thus, we may construct from $\mathcal{D}$, an $N\times K$ real-valued matrix
$$\mathcal{X}:= (x_{n,k})_{1\le n \le N,\, 1\le k \le K}.$$

We may consider $\mathcal{D}$ to be a *dataset* with $N$ fixed *observations* $\mathbf{x}_1, \ldots, \mathbf{x}_N$. These observations are $K$-tuples, where each entry in the tuple corresponds to the value the given observation takes on with respect to the $k^{th}$ *variable of interest*.

Suppose also that we are given a vector $\mathcal{Y} = (y_1, \ldots, y_N) \in \mathbb{R}^N$. We may think of $\mathcal{Y}$ as *labels* corresponding to each observation $(\mathbf{x}_n, y_n)$. That is, we represent the vector $\mathbf{x}_n$ by one scalar $y_n \in \mathbb{R}$. For example, our dataset could consist of variables describing the characteristics of a home for sale (e.g. sq. footage, number of bedrooms, etc.), while the label could simply be the home price.

With respect to the data $(\mathbf{x}_n ,y_n)$, we assume that there exists a *data-generating* function $f:\mathbb{R}^K \to \mathbb{R}$, where $f(\mathbf{x}_n)$ is the *true* label with respect to $\mathbf{x}_n$. This function is usually **unknown** to us. In practice, there is typically some form of measurement error with respect to the data. As an example, any data collected from some sort of physical sensor will contain small measurement error. With this in mind, we assume that the label data that we measure, $\mathcal{Y} = (y_1, \ldots y_N)$, is a *noisy* output of this data-generating function. Many texts write shorthand notation for this concept. To make this more concrete for the first-time reader, first recall that $\mathcal{B}(\R)$ is the Borel $\sigma$-algebra on $\R$. Now suppose that $(\Omega, \mathcal{F}, \mathbb{P})$ is a probability space with $\mathcal{F}/\mathcal{B}(\R)$-measurable functions 

$$
\epsilon_1, \ldots, \epsilon_N: \Omega \to \R
$$

such that for $A \in \mathcal{B}(\R)$,

$$   
    \mathbb{P}(\epsilon_n^{-1}(A)) = \int_A \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left(-\frac{t^2}{2\sigma^2}\right)\:dt, \quad 1 \le n \le N,
$$

where $\sigma^2 >0$ is known to us. Equivalently, 
$$
    \mathbb{P}\circ \epsilon_n^{-1} = \mathcal{N}(0, \sigma^2), \quad 1 \le n \le N.
$$

Thus, the random variables $\epsilon_n$ are identically distributed. Moreover, let us assume the $\epsilon_n$ are independent; i.e., for $A_1, \ldots, A_N \in \mathcal{B}(\R)$,

$$
    \mathbb{P}\left(\bigcap^N_{n=1}\epsilon_n^{-1}(A_n)\right) = \prod^N_{n=1} \mathbb{P}(\epsilon_n^{-1}(A_n)).
$$

Thus, the $\epsilon_n$ are independent and identically distributed (i.i.d.) zero-centered Gaussian noise. Note here that the probability space is entirely abstract. We just need a probability space that supports these $N$ i.i.d. Gaussians. Now define the random variables

$$ Y_n: \Omega \to \R, \quad Y_n(\omega) = f(\mathbf{x}_n) + \epsilon_n(\omega)$$
for each $1 \le n \le N$. Thus,
$$
    Y_n \sim \mathcal{N}(f(\mathbf{x}_n), \sigma^2).
$$

Observe that the $Y_n$ are independent. Indeed, for any Borel set $A \subseteq \R$ and $t \in \R$, it follows that $A +t = \{a+t: a \in A\} \in \mathcal{B}(\R)$. In particular, for $A_1 \ldots, A_N \in \mathcal{B}(\R)$, we have that

$$
\mathbb{P}\left(\bigcap^N_{n=1}Y_n^{-1}(A_n)\right) = \mathbb{P}\left(\bigcap^N_{n=1}\epsilon_n^{-1}(A_n - f(\mathbf{x}_n))\right) = \prod^N_{n=1} \mathbb{P}(\epsilon_n^{-1}(A_n - f(\mathbf{x}_n))) =  \prod^N_{n=1} \mathbb{P}(Y_n^{-1}(A_n)). 
$$

Define the random vector
$$
    \mathbf{Y}: \Omega \to \R^N, \quad \mathbf{Y}(\omega) = (Y_1(\omega), \ldots, Y_N(\omega)).
$$

We assume that our observed labels are a realization of $\mathbf{Y}$. That is, for some $\omega_0 \in \Omega$, we have

$$
\mathcal{Y} = \mathbf{Y}(\omega_0) = (Y_1(\omega_0), \ldots, Y_N(\omega_0)) = (y_1, \ldots, y_N).
$$

**Our Goal:** Find a function $\widehat{f}: \mathbb{R}^K \to \mathbb{R}$ such that $\widehat{f}$ approximates $f$ "well". Since $f$ is unknown, we cannot directly evaluate the discrepancy between $f$ and $\widehat f$. Instead, we introduce the notion of a *loss function*, which allows us to quantify the discrepancy between our predictions and the noisy observed labels. We will discuss this concept in greater detail in the following section.

## Linear Regression

We again consider our data set $\mathcal{X} = (\mathbf{x}_n)^N_{n=1}$, $\mathbf{x}_n \in \R^K$ for $1 \le n \le N$, with labels $\mathcal{Y} \in \R^N$ as defined above.

Now that we have discussed the main problem formulation for regression, we restrict our attention to data-generating functions $f:\mathbb{R}^K \to \R$ belonging to the class of affine functions on $\mathbb{R}^K$. In particular, we assume there exists $\mathbf{\theta}_* = (\theta_{*,1}, \ldots, \theta_{*,K}) \in \R^K$ and $b_* \in \R$ such that

$$f(\mathbf{x}) = \langle \mathbf{x}, \mathbf{\theta}_* \rangle_{\R^K} + b_*,$$

where $\langle \cdot, \cdot\rangle_{\R^K}$ is the Euclidean inner-product. This inner product is also referred to as the *dot product*. Here, $\mathbf{\theta}_*$ and $b_*$ are unknown parameters that we wish to estimate. 

To make our lives easier, we may consider the data to be of the form $\tilde{\mathbf{x}}_n:= (1, x_{n,1}, \ldots, x_{n,K})$. Appending $1$ to the original tuples allows us to remove the $b_*$ term in the formulation above. Indeed, consider the vector $\tilde{\mathbf{\theta}}_*:= (b_*, \theta_{*,1}, \ldots, \theta_{*,K}) \in \R^{K+1}$. Then we define $\tilde{f}:\R^{K+1} \to \R$ as

$$\tilde{f}(\mathbf{x}) = \langle \mathbf{x},\tilde{\mathbf{\theta}}_* \rangle_{\R^{K+1}}.$$

This gives

$$
    f(\mathbf{x}_n) = \langle \mathbf{x}_n, \mathbf{\theta}_* \rangle_{\R^K} + b_* = \langle \tilde{\mathbf{x}}_n, \tilde{\mathbf{\theta}}_* \rangle_{\R^{K+1}} = \tilde{f}(\tilde{\mathbf{x}}_n)
$$
for each $1 \le n \le N$. Define

$$
    \iota: \R^{K} \to \R^{K+1}, \quad \iota(\mathbf{x}) = (1, \mathbf{x}).
$$

Then it is clear that $f = \tilde{f} \circ \iota$, and thus, the augmented formulation agrees with the original function on every input under the natural identification. To ease notation, we will no longer write the tilde symbol above the data and the function $f$ and relabel the dimension $K+1$ to $K$. 


We see that $f \in (\R^{K})^*$, the dual of $\R^{K}$, and so the problem then becomes: find a linear functional in the dual of $\R^{K}$ that approximates the data-generating function $f$.

Towards this end, we observe that by the independence of the $Y_1, \ldots, Y_N$, the joint density of $\mathbf{Y}$ evaluated at $\mathcal{Y}$ is given by

$$
p(\mathcal{Y}| \mathcal{X}, \mathbf{\theta}_* ) = \prod^N_{n=1} p(y_n| \mathbf{x}_n, \mathbf{\theta}_*) = \prod^N_{n=1}\mathcal{N}(y_n|\langle \mathbf{x}_n, \mathbf{\theta}_* \rangle_{\R^K}, \sigma^2) = \prod^N_{n=1} \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left(-\frac{(y_n-\langle \mathbf{x}_n, \mathbf{\theta}_*\rangle_{\R^K})^2}{2\sigma^2}\right).
$$ 

Since $\mathbf{\theta}_*$ is unknown to us, we would like to find some parameter $\mathbf{\theta} \in \R^K$ that gives us a good approximation to $f$. For each candidate parameter, we generate a new probability density $p(\cdot | \mathcal{X}, \mathbf{\theta})$. Fixing our observed labels $\mathcal{Y}$, we define the likelihood function

$$
    \mathcal{L}: \R^K \to (0, \infty), \quad \mathcal{L}(\mathbf{\theta}) = p(\mathcal{Y}| \mathcal{X}, \mathbf{\theta}) = \prod^N_{n=1} \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left(-\frac{(y_n-\langle \mathbf{x}_n, \mathbf{\theta}\rangle_{\R^K})^2}{2\sigma^2}\right).
$$

### Maximizing the Likelihood